# Week 13 - Music Generation using Deep Learning: Concepts & Representation

**Objectives of the experiment (Weeks 13-14)**
* Understand how deep learning models can generate music.
* Explore **Char-RNN** and its role in music generation using **ABC notation**.
* Pre-process ABC datasets, build an RNN for music sequence learning.
* Implement **MIDI-based** music generation and generate tabla-style sequences.

## a. Real-world problem and music representation
A music-generation model learns patterns from existing music and then produces new sequences. Uses: composition aid, game soundtracks, musical assistance, style experiments.

The core trick: **music is a sequence**, so we can treat it like text - show the model the past, ask for the next element (a character, a note, a MIDI event), and repeat.

| Representation | What it is | Good for |
|---|---|---|
| **ABC notation** | music as readable text: notes (`C D E G`), rhythm, meter, key | character-level language models (Char-RNN) |
| **MIDI** | musical *events*: pitch, note-on/off times, velocity, instrument. Not an audio recording | note-level models; can be played by any synthesiser |

## b. Char-RNN with ABC notation
A **character-level RNN** reads text one character at a time. An **LSTM** layer keeps a memory of what came before, and the output layer predicts a probability for *every possible next character*. After training, we pick a character from that distribution, append it, and feed the extended text back in: this loop is how new music is *generated*.

```
 C D E G | G E D  ->  [Embedding] -> [LSTM] -> [Dense softmax] -> next char: " "
```

The model learns which characters and note patterns commonly follow each other (e.g. a bar line `|` usually appears after a certain number of notes).

## Demo: reading ABC notation
An ABC tune has a **header** (metadata fields) and a **body** (the notes).

| Field | Meaning |
|---|---|
| `X:` | tune number |
| `T:` | title |
| `M:` | meter (4/4 = four quarter-note beats per bar) |
| `L:` | default note length (1/8 = eighth note) |
| `K:` | key (C major) |
| body | notes `C D E ...`, `|` = bar line, `G2` = G held for 2 units |

In [1]:
abc = """X:1
T:Simple Rhythm
M:4/4
L:1/8
K:C
C D E G | G E D C |
C C G G | A A G2 |
"""

header = [l for l in abc.splitlines() if len(l) > 1 and l[1] == ":"]
body = [l for l in abc.splitlines() if not (len(l) > 1 and l[1] == ":")]
print("Header fields:", header)
print("Body lines   :", body)

bars = [b.strip() for line in body for b in line.split("|") if b.strip()]
print("\nBars found:", len(bars))
for i, b in enumerate(bars, 1):
    print(f"  bar {i}: {b!s:12} -> {len(b.split())} notes/units")

Header fields: ['X:1', 'T:Simple Rhythm', 'M:4/4', 'L:1/8', 'K:C']
Body lines   : ['C D E G | G E D C |', 'C C G G | A A G2 |']

Bars found: 4
  bar 1: C D E G      -> 4 notes/units
  bar 2: G E D C      -> 4 notes/units
  bar 3: C C G G      -> 4 notes/units
  bar 4: A A G2       -> 3 notes/units


## Demo: why a model must predict the "next character"
For the string `"C D E G"`, each position's target is simply the **next** character. This is the *self-supervised* training signal - no manual labels needed.

In [2]:
text = "C D E G | G E D C |"
print("input char  -> target char")
for a, b in zip(text[:-1], text[1:]):
    print(f"   {a!r:5}   ->   {b!r}")

input char  -> target char
   'C'     ->   ' '
   ' '     ->   'D'
   'D'     ->   ' '
   ' '     ->   'E'
   'E'     ->   ' '
   ' '     ->   'G'
   'G'     ->   ' '
   ' '     ->   '|'
   '|'     ->   ' '
   ' '     ->   'G'
   'G'     ->   ' '
   ' '     ->   'E'
   'E'     ->   ' '
   ' '     ->   'D'
   'D'     ->   ' '
   ' '     ->   'C'
   'C'     ->   ' '
   ' '     ->   '|'


## Demo: probabilities and sampling
The network outputs a probability for each character. Two ways to choose the next one:
* **argmax** - always take the most likely (repetitive, boring),
* **sampling** - draw randomly according to the probabilities (more variety). The record uses sampling.

A toy example with an artificial distribution:

In [3]:
import numpy as np
rng = np.random.default_rng(0)

chars = ["C", "D", "E", "G", "|"]
probs = np.array([0.40, 0.20, 0.15, 0.15, 0.10])

print("argmax choice:", chars[int(np.argmax(probs))])
samples = rng.choice(chars, size=15, p=probs)
print("15 sampled choices:", " ".join(samples))
print("Observed frequencies:", {c: round(float(np.mean(samples == c)), 2) for c in chars})

argmax choice: C
15 sampled choices: E C C C G | E E D | G C G C E
Observed frequencies: {'C': 0.33, 'D': 0.07, 'E': 0.27, 'G': 0.2, '|': 0.13}


## Summary & next week
* Music can be modelled as a sequence of characters (ABC) or events (MIDI).
* Char-RNN = Embedding -> LSTM -> softmax, trained to predict the next element, generating by repeated sampling.
* **Week 14** builds this for real: ABC data preparation, the many-to-many LSTM, generation, and a MIDI-based model for tabla-style music.